In [ ]:
import pandas as pd
import numpy as np

DATA_FILE = '../data/scenario_6_marina_features_50ms_final.csv'
df = pd.read_csv(DATA_FILE)

df.sort_values(['video_id', 'iteration', 'timestamp'], inplace=True)

print("Generating Advanced Network Features...")

#we don't calculate trends across different video sessions
g = df.groupby(['video_id', 'iteration'])

# Exponential Moving Averages (EMA) 
# Simulates the switch keeping a 'running average' state
# span=20 roughly equals 1 second of history (20 * 50ms)
df['bw_ema'] = g['bwe'].transform(lambda x: x.ewm(span=20, adjust=False).mean())
df['jitter_ema'] = g['jitter'].transform(lambda x: x.ewm(span=20, adjust=False).mean())
df['iat_ema'] = g['iat_sum'].transform(lambda x: x.ewm(span=20, adjust=False).mean())

#  Lag Features (What happened 50ms ago?)
cols_to_lag = ['bwe', 'jitter', 'packet_count']
for col in cols_to_lag:
    df[f'{col}_prev'] = g[col].shift(1) # 1 step back

#Rate of Change (Acceleration) 
# Is bandwidth dropping fast?
df['bw_change'] = df['bwe'] - df['bwe_prev']

df.dropna(inplace=True)

print(f"Feature Engineering Complete. Final Shape: {df.shape}")
df.head()

In [ ]:
LOOKAHEAD_STEPS = 10       # Look 500ms into the future
DROP_THRESHOLD = -200      # Buffer dropping by > 200ms

# Calculate future slope
df['future_buffer'] = df.groupby(['video_id', 'iteration'])['buffer_level_ms'].shift(-LOOKAHEAD_STEPS)
df['buffer_slope'] = df['future_buffer'] - df['buffer_level_ms']

df.dropna(subset=['future_buffer'], inplace=True)

# Binary label: At_Risk if buffer is dropping significantly, Steady otherwise
df['qoe_state'] = np.where(
    (df['buffer_slope'] < DROP_THRESHOLD),
    'At_Risk',
    'Steady'
)

print("Target Class Distribution:")
print(df['qoe_state'].value_counts(normalize=True))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report


features = [
    # Instant values
    'packet_count', 'ps_sum', 'ps2_sum', 'ps3_sum', 
    'iat_sum', 'iat2_sum', 'iat3_sum',
    'jitter',
    # Historical Context 
    #'bw_ema',
    #'jitter_ema', 'iat_ema',
    #'bwe_prev', 
    #'jitter_prev',
    #'bw_change',
]

X = df[features]
y = df['qoe_state']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=12, 
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred))

In [ ]:
import matplotlib.pyplot as plt

importances = rf_model.feature_importances_

feature_importances = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importances['Feature'], feature_importances['Importance'], color='skyblue')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importances')
plt.gca().invert_yaxis()
plt.show()

print(feature_importances)

In [ ]:
print("X", X.shape)
print("X", X.head())
np.save('np_data.npy', X.to_numpy())

y_dummies = pd.get_dummies(y)
np.save('np_dummies.npy', y_dummies.to_numpy())

print("Files 'np_data.npy' and 'np_dummies.npy' have been created.")

Capture "ON/OFF" Burst Patterns
The Problem: Video traffic is not constant; it comes in bursts (ON periods) followed by silence (OFF periods).

Maybe Create features that specifically measure the "Silence"?